## Setup

### Imports

In [1]:
import anndata as ad
import mudata as md
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

In [2]:
# load precursor and protein data from diann with alphapepttools functions

In [3]:
# load h5ad data saved by alphapepttools
prec_adata = ad.read_h5ad("../data/albrecht.precursors.h5ad")
prot_adata = ad.read_h5ad("../data/albrecht.proteins.h5ad")

In [4]:
msdata = md.MuData(
    # These are the raw data levels
    {
        "protein_level": prot_adata,
        #"peptide_level": ad.AnnData(...),
        "precursor_level": prec_adata,
    },
    # This stores the feature mapping as adjacency matrix of a DAG
    #varp = csr_matrix(...)
)

/Users/mcthielert/miniforge3/envs/msmudata_dev/lib/python3.14/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/Users/mcthielert/miniforge3/envs/msmudata_dev/lib/python3.14/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [6]:
# Build the precursor -> protein adjacency matrix (varp)
n_prec = len(prec_adata.var)
n_prot = len(prot_adata.var)
n_total = n_prec + n_prot 

# Each precursor maps to one protein group via Protein.Group column
prot_index = pd.Index(prot_adata.var.index)
prec_protein_groups = prec_adata.var["Protein.Group"]

# Precursors are rows 0:n_prec, proteins are rows n_prec:n_total
row_idx = np.arange(n_prec)
col_idx = prot_index.get_indexer(prec_protein_groups) + n_prec  # offset into protein block

assert (col_idx >= n_prec).all(), "Some precursors don't map to any protein group!"

mapping_matrix_varp = csr_matrix(
    (np.ones(len(row_idx), dtype=np.float32), (row_idx, col_idx)),
    shape=(n_total, n_total),
)

print(f"Mapping matrix shape: {mapping_matrix_varp.shape}")  
print(f"Non-zero entries: {mapping_matrix_varp.nnz}")     

Mapping matrix shape: (17250, 17250)
Non-zero entries: 15089


In [7]:
msdata

MuData object with n_obs × n_vars = 1801 × 17250
  2 modalities
    protein_level:	1801 x 2161
    precursor_level:	1801 x 15089
      var:	'Run', 'Stripped.Sequence', 'Precursor.Charge', 'RT', 'RT.Start', 'RT.Stop', 'IM', 'Protein.Group', 'Protein.Ids', 'Genes', 'MS2.Scan', 'CScore', 'Q.Value', 'Global.Q.Value', 'Global.PG.Q.Value', 'Lib.Q.Value', 'Lib.PG.Q.Value'

In [ ]:
#msdata.varm["precursor_to_protein"] = mapping_matrix_varm
msdata.varp["precursor_to_protein"] = mapping_matrix_varp

In [36]:
msdata

MuData object with n_obs × n_vars = 1801 × 17250
  varp:	'precursor_to_protein'
  2 modalities
    protein_level:	1801 x 2161
    precursor_level:	1801 x 15089
      var:	'Run', 'Stripped.Sequence', 'Precursor.Charge', 'RT', 'RT.Start', 'RT.Stop', 'IM', 'Protein.Group', 'Protein.Ids', 'Genes', 'MS2.Scan', 'CScore', 'Q.Value', 'Global.Q.Value', 'Global.PG.Q.Value', 'Lib.Q.Value', 'Lib.PG.Q.Value'

In [37]:
prec_adata.var

,Run,Stripped.Sequence,Precursor.Charge,RT,RT.Start,RT.Stop,IM,Protein.Group,Protein.Ids,Genes,MS2.Scan,CScore,Q.Value,Global.Q.Value,Global.PG.Q.Value,Lib.Q.Value,Lib.PG.Q.Value
AAAAAAALQAK2,20240313_OA3_Evo07_ViAl_SA_FAIMS40_IO14_A556_M...,AAAAAAALQAK,2.0,3.63931,3.56613,3.71280,0.0,P36578,P36578;H3BU31;H3BM89,RPL4,22131.0,0.999087,0.000222,3.774200e-05,0.000516,1.827500e-05,0.000472
AAAATGTIFTFR2,20250418_OA3_Evo07_ViAl_SA_FAIMS40_IO44_A556_M...,AAAATGTIFTFR,2.0,7.75061,7.67728,7.82353,0.0,P05154,P05154;G3V5I3,SERPINA5,47052.0,0.999870,0.000207,1.352830e-06,0.000516,6.550540e-07,0.000472
AAAATGTIFTFR3,20250429_OA3_Evo03_ViAl_SA_FAIMS40_IO46_A556_M...,AAAATGTIFTFR,3.0,7.96541,7.89213,8.03826,0.0,P05154,P05154;G3V5I3,SERPINA5,48303.0,0.994502,0.000733,4.735500e-03,0.000516,4.660040e-03,0.000472
AAAFEEQENETVVVK2,20240312_OA3_Evo07_ViAl_SA_FAIMS40_IO18_A556_M...,AAAFEEQENETVVVK,2.0,5.57329,5.50005,5.64629,0.0,Q9Y490,Q9Y490,TLN1,34103.0,0.999816,0.000260,1.812260e-06,0.000516,8.775140e-07,0.000472
AAAFLGDIALDEEDLR3,20250429_OA3_Evo03_ViAl_SA_FAIMS40_IO46_A556_M...,AAAFLGDIALDEEDLR,3.0,9.62869,9.55572,9.70213,0.0,P13497,P13497;B7ZKR5;P13497-2;P13497-3;P13497-4;P1349...,BMP1,58598.0,0.895719,0.008586,4.735500e-03,0.000516,1.161820e-03,0.000472
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YYVTIIDAPGHR3,20240329_OA3_Evo07_ViAl_SA_FAIMS40_IO16_A556_M...,YYVTIIDAPGHR,3.0,6.43221,6.36019,6.50453,0.0,P68104;Q5VTE0,P68104;Q5VTE0;Q5JR01;A0A087WV01;A0A087WVQ9;A6P...,EEF1A1;EEF1A1P5,39463.0,0.999051,0.000290,9.624450e-06,0.000516,4.660240e-06,0.000472
YYVTIIDAPGHRDFIK4,20250123_OA3_Evo07_ViAl_SA_FAIMS40_IO36_A556_M...,YYVTIIDAPGHRDFIK,4.0,6.99931,6.92584,7.07198,0.0,P68104;Q5VTE0,P68104;Q5VTE0;Q5JR01;A0A087WV01;A0A087WVQ9;A6P...,EEF1A1;EEF1A1P5,42459.0,0.999954,0.000087,2.599950e-05,0.000516,1.258920e-05,0.000472
YYWGGQYTWDM(UniMod:35)AK2,20250403_OA3_Evo07_ViAl_SA_FAIMS40_IO44_A556_M...,YYWGGQYTWDMAK,2.0,7.66796,7.59481,7.74104,0.0,P02675,P02675;D6REL8,FGB,46410.0,0.986862,0.000680,1.337110e-06,0.000516,6.474380e-07,0.000472
YYWGGQYTWDMAK2,20250418_OA3_Evo07_ViAl_SA_FAIMS40_IO44_A556_M...,YYWGGQYTWDMAK,2.0,8.46702,8.39425,8.54041,0.0,P02675,P02675;D6REL8,FGB,51504.0,0.997828,0.000980,4.572830e-07,0.000516,2.214200e-07,0.000472


In [38]:
def get_unique_mappings(psm_table: pd.DataFrame, feature_level_names: list[str]) -> pd.DataFrame:
    """
    Get unique mappings from PSM table for specified feature levels.
    
    Parameters:
    - psm_table: DataFrame containing PSM data with columns for each feature level.
    - feature_level_names: List of column names corresponding to feature levels (e.g., ["Precursor", "Protein"]).
    
    Returns:
    - DataFrame with unique mappings between the specified feature levels.
    """
    # Select only the relevant columns for mapping
    mapping_df = psm_table[feature_level_names].drop_duplicates()
    
    return mapping_df

In [39]:
test_map = get_unique_mappings(prec_adata.var, ["Protein.Group"])

In [40]:
test_map

,Protein.Group
AAAAAAALQAK2,P36578
AAAATGTIFTFR2,P05154
AAAFEEQENETVVVK2,Q9Y490
AAAFLGDIALDEEDLR3,P13497
AAAGEFADDPC(UniMod:4)SSVK2,P35221;P35221-2
...,...
YPVVPVHLDTTI2,P12724
YQAVTATLEEK2,P40429
YQFFVYLQEGK2,Q96S96
YREWHHFLVVNMK4,P30086


In [41]:
def sparse_matrix_mapping(mapping_df: pd.DataFrame) -> csr_matrix:
    """
    Create a sparse matrix mapping from source to target features.
    
    Parameters:
    - mapping_df: DataFrame containing unique mappings between source and target features.

    Returns:
    - Sparse matrix (csr_matrix) representing the mapping from source to target features.
    """
    # Get unique source and target features
    unique_sources = mapping_df[source_col].unique()
    unique_targets = mapping_df[target_col].unique()
    
    # Create index mappings for sources and targets
    source_index = {feature: idx for idx, feature in enumerate(unique_sources)}
    target_index = {feature: idx for idx, feature in enumerate(unique_targets)}
    
    # Prepare data for sparse matrix construction
    row_idx = mapping_df[source_col].map(source_index).values
    col_idx = mapping_df[target_col].map(target_index).values
    data = np.ones(len(mapping_df), dtype=np.float32)
    
    # Create sparse matrix
    sparse_matrix = csr_matrix((data, (row_idx, col_idx)), shape=(len(unique_sources), len(unique_targets)))
    
    return sparse_matrix

In [ ]:
matrix_varp = sparse_matrix_mapping(test_map, sourc)

NameError: name 'source_col' is not defined

In [ ]:
matrix_varp

In [ ]:

# Check for unmapped precursors (0 means no match)
unmapped = prec_adata.var[col_idx == 0]
print(f"Unmapped precursors: {len(unmapped)}")
print(unmapped["Protein.Group"].head())  # inspect why they don't match

# Build CSR: shape (n_prec x n_prot)
row_idx = np.arange(len(prec_adata.var))

mapping_matrix = csr_matrix(
    (np.ones(len(row_idx[col_idx >= 0]), dtype=np.float32),
     (row_idx[col_idx >= 0], col_idx[col_idx >= 0])),
    shape=(len(prec_adata.var), len(prot_adata.var)),
)

In [ ]:
def sparse_matrix_mapping(mapping_df: pd.DataFrame) -> csr_matrix:
    """
    Create a sparse matrix mapping from source to target features.
    
    Parameters:
    - mapping_df: DataFrame containing unique mappings between source and target features.

    Returns:
    - Sparse matrix (csr_matrix) representing the mapping from source to target features.
    """
    # Get unique source and target features
    unique_sources = mapping_df[source_col].unique()
    unique_targets = mapping_df[target_col].unique()
    
    # Create index mappings for sources and targets
    source_index = {feature: idx for idx, feature in enumerate(unique_sources)}
    target_index = {feature: idx for idx, feature in enumerate(unique_targets)}
    
    # Prepare data for sparse matrix construction
    row_idx = mapping_df[source_col].map(source_index).values
    col_idx = mapping_df[target_col].map(target_index).values
    data = np.ones(len(mapping_df), dtype=np.float32)
    
    # Create sparse matrix
    sparse_matrix = csr_matrix((data, (row_idx, col_idx)), shape=(len(unique_sources), len(unique_targets)))
    
    return sparse_matrix

In [17]:
mapping_matrix_varp.indices

array([16069, 15553, 15553, ..., 15477, 15477, 15477],
      shape=(15089,), dtype=int32)

In [18]:
msdata.var_names

Index(['A0A024R6I7;A0A0G2JRN3', 'A0A075B6H7', 'A0A075B6I0', 'A0A075B6I9',
       'A0A075B6J2', 'A0A075B6J9', 'A0A075B6K4', 'A0A075B6K5',
       'A0A075B6P5;P01615', 'A0A075B6Q5',
       ...
       'YYTVFDR2', 'YYTVFDRDNNR2', 'YYTYLIM(UniMod:35)NK2', 'YYTYLIMNK2',
       'YYVTIIDAPGHR2', 'YYVTIIDAPGHR3', 'YYVTIIDAPGHRDFIK4',
       'YYWGGQYTWDM(UniMod:35)AK2', 'YYWGGQYTWDMAK2', 'YYWGGQYTWDMAK3'],
      dtype='object', length=17250)

In [22]:
pd.DataFrame.sparse.from_spmatrix(mapping_matrix_varp, index=msdata.var_names, columns=msdata.var_names).reindex(msdata.var_names,columns=msdata.var_names)

,A0A024R6I7;A0A0G2JRN3,A0A075B6H7,A0A075B6I0,A0A075B6I9,A0A075B6J2,A0A075B6J9,A0A075B6K4,A0A075B6K5,A0A075B6P5;P01615,A0A075B6Q5,...,YYTVFDR2,YYTVFDRDNNR2,YYTYLIM(UniMod:35)NK2,YYTYLIMNK2,YYVTIIDAPGHR2,YYVTIIDAPGHR3,YYVTIIDAPGHRDFIK4,YYWGGQYTWDM(UniMod:35)AK2,YYWGGQYTWDMAK2,YYWGGQYTWDMAK3
A0A024R6I7;A0A0G2JRN3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
A0A075B6H7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
A0A075B6I0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
A0A075B6I9,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
A0A075B6J2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YYVTIIDAPGHR3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
YYVTIIDAPGHRDFIK4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
YYWGGQYTWDM(UniMod:35)AK2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
YYWGGQYTWDMAK2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [15]:
mapping_matrix_varp.sum()

np.float32(15089.0)

In [16]:
prot_adata

AnnData object with n_obs × n_vars = 1801 × 2161

In [17]:
prec_adata

AnnData object with n_obs × n_vars = 1801 × 15089
    var: 'Run', 'Stripped.Sequence', 'Precursor.Charge', 'RT', 'RT.Start', 'RT.Stop', 'IM', 'Protein.Group', 'Protein.Ids', 'Genes', 'MS2.Scan', 'CScore', 'Q.Value', 'Global.Q.Value', 'Global.PG.Q.Value', 'Lib.Q.Value', 'Lib.PG.Q.Value'

In [ ]:
# Build the precursor -> protein adjacency matrix (varp)
# Each precursor maps to one protein group via Protein.Group column
prot_index = pd.Index(prot_adata.var.index)
prec_protein_groups = prec_adata.var["Protein.Group"]


# Row indices: precursors, Col indices: proteins
row_idx = np.arange(len(prec_adata.var))
col_idx = prot_index.get_indexer(prec_protein_groups)

# Sanity check: all precursors should map to a known protein group
assert (col_idx >= 0).all(), "Some precursors don't map to any protein group!"

mapping_matrix = csr_matrix(
    (np.ones(len(row_idx), dtype=np.float32), (row_idx, col_idx)),
    shape=(len(prec_adata.var), len(prot_adata.var)),
)
print(f"Mapping matrix shape: {mapping_matrix.shape} (precursors x proteins)")
print(f"Non-zero entries: {mapping_matrix.nnz}")


NameError: name 'pd' is not defined

In [ ]:

# Row indices: precursors, Col indices: proteins
row_idx = np.arange(len(prec_adata.var))
col_idx = prot_index.get_indexer(prec_protein_groups)

# Sanity check: all precursors should map to a known protein group
assert (col_idx >= 0).all(), "Some precursors don't map to any protein group!"

mapping_matrix = csr_matrix(
    (np.ones(len(row_idx), dtype=np.float32), (row_idx, col_idx)),
    shape=(len(prec_adata.var), len(prot_adata.var)),
)
print(f"Mapping matrix shape: {mapping_matrix.shape} (precursors x proteins)")
print(f"Non-zero entries: {mapping_matrix.nnz}")